In [1]:
import sys
from pathlib import Path

# Add src/ to sys.path regardless of where Jupyter is launched from
for _candidate in [Path().resolve().parent / "src", Path().resolve() / "src"]:
    if _candidate.exists() and str(_candidate) not in sys.path:
        sys.path.insert(0, str(_candidate))
        print(f"Added to sys.path: {_candidate}")
        break

Added to sys.path: C:\Users\loren\Documents\Postdoc\Compressed_sensing\compressed_sensing_bioacoustics\src


In [2]:
from compress import CS
import time
import os
from pathlib import Path
import datetime
from codecarbon import EmissionsTracker
import psutil
import threading
import gc

c:\Users\loren\anaconda3\envs\cs_HeAims\Lib\site-packages\codecarbon\core\gpu.py:4: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml


In [3]:
def monitor_resources(stop_event, cpu_usage, mem_usage, sampling_interval=0.1):
    """
    Monitor CPU and memory usage of the current process at regular intervals.
    - cpu_usage: list to store CPU usage (%) over time
    - mem_usage: list to store memory usage (MB) over time
    """
    process = psutil.Process(os.getpid())
    
    while not stop_event.is_set():
        # CPU percent since last call (per process)
        cpu = process.cpu_percent(interval=None)
        # Memory usage (RSS in MB)
        mem = process.memory_info().rss / (1024 ** 2)
        
        cpu_usage.append(cpu)
        mem_usage.append(mem)
        
        time.sleep(sampling_interval)

In [4]:
gc.collect()


13

In [5]:
#parameters species
folder_audio="C:/Users/loren/Documents/Postdoc/Compressed_sensing/Data/Bats/Audio"
folder_saved="C:/Users/loren/Documents/Postdoc/Compressed_sensing/Data/Bats/Compressed_Audio"
folder_tracking="C:/Users/loren/Documents/Postdoc/Compressed_sensing/Data/Bats/tracking"
compression_rate=0.1
sample_rate=256000       # raw bat recording sample rate (256 kHz)

#parameter cs
# 1024 samples @ 256 kHz = 4 ms per frame — matches one bat call timescale
# and is equivalent to n_fft=512 at the 128 kHz downsampled rate used by the mel-spectrogram
frame_size=1024
overlap=0.5
compression_mode="compression" #compression or reconstruction

In [7]:
os.cpu_count()

20

In [8]:
n_jobs=os.cpu_count() - 1
n_jobs=10

In [9]:
cs=CS(folder_audio, folder_saved, sample_rate, frame_size, overlap, compression_rate=compression_rate, n_jobs=n_jobs)

In [10]:
starting = time.time()
if compression_mode=="compression": 
    print(" compression mode: ", compression_mode)
    #times=cs.compress_folder()
    times=cs.compress_folder_legacy()
else :
    print(" reconstruction mode : " , compression_mode)
    # max_iter=200: more iterations for high-frequency bat signals at 256 kHz
    # sparsity=None: uses heuristic max(8, min(M//2, N//5)) → ~102 for M=204, N=1024
    #times=cs.reconstruction(alpha=1e-7, max_iter=200)
    times=cs.reconstruction_legacy(alpha=1e-7)
        
execution_time_baseline = time.time() - starting

 compression mode:  compression


Compressing audio segments: 100%|██████████| 1961/1961 [00:00<00:00, 175392.48segment/s]


In [14]:
save_path=f'{folder_tracking}/time_execution_{compression_mode}_{compression_rate}.txt'
with open(save_path, "w") as f:
    f.write(f"time execution: {execution_time_baseline}")

print(execution_time_baseline)

188.02443528175354
